**Ch121a | Module 3: Periodic DFT**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/Module3_Periodic-DFT/notebooks/04b_adsorption_energy.ipynb)

# Notebook 4b: Advanced Example: CO vs. H Adsorption Energy on Cu(100)

---

## Learning Objectives

- Build a periodic slab model of a Cu(100) surface
- Run adsorbate+slab and clean-slab VASP calculations
- Compute adsorption energies of CO and H on Cu(100)
- Understand the hollow, bridge, and top adsorption sites
- Compare CO vs. H adsorption energies and interpret the results

## 1. Adsorption Energy: Concept

The adsorption energy is defined as:

$$E_{\rm ads} = E_{\rm slab+ads} - E_{\rm slab} - E_{\rm ads(gas)}$$

where:
- $E_{\rm slab+ads}$ — total energy of the slab with the adsorbate
- $E_{\rm slab}$ — total energy of the clean slab (same geometry, same number of layers)
- $E_{\rm ads(gas)}$ — total energy of the isolated adsorbate molecule in vacuum

A **negative** $E_{\rm ads}$ means exothermic (favorable) adsorption.

| Adsorbate | Typical $E_{\rm ads}$ on Cu(100) (PBE) | Notes |
|-----------|----------------------------------------|-------|
| CO | −0.5 to −0.8 eV | Top site preferred; PBE slightly underestimates exp. |
| H | −2.4 to −2.7 eV | Hollow site preferred; strong chemisorption |

> **Note**: These are PBE values. vdW corrections (IVDW=12) improve CO adsorption energies; H adsorption is less sensitive to dispersion.

## 2. Slab Model: Cu(100)

We use a **4-layer Cu(100) slab** with a (2×2) lateral supercell (4 Cu atoms per layer, 16 total) and ~15 Å of vacuum to decouple periodic images.

### Cu(100) POSCAR (clean slab, 4 layers, 2×2)

```
Cu(100) 2x2 4-layer slab
1.0
   7.234   0.000   0.000
   0.000   7.234   0.000
   0.000   0.000  20.000
Cu
16
Selective dynamics
Direct
  0.000  0.000  0.000  F F F
  0.500  0.000  0.000  F F F
  0.000  0.500  0.000  F F F
  0.500  0.500  0.000  F F F
  0.250  0.250  0.091  F F F
  0.750  0.250  0.091  F F F
  0.250  0.750  0.091  F F F
  0.750  0.750  0.091  F F F
  0.000  0.000  0.182  T T T
  0.500  0.000  0.182  T T T
  0.000  0.500  0.182  T T T
  0.500  0.500  0.182  T T T
  0.250  0.250  0.273  T T T
  0.750  0.250  0.273  T T T
  0.250  0.750  0.273  T T T
  0.750  0.750  0.273  T T T
```

**Selective dynamics**: bottom two layers are frozen (`F F F`); top two layers are relaxed (`T T T`) to mimic a semi-infinite bulk while keeping computational cost manageable.

The lattice constant for Cu is 3.617 Å, so the 2×2 cell is 2 × 3.617 Å ≈ 7.234 Å.

### Adsorption Sites on Cu(100)

```
    ·─────·─────·
    │  H  │  H  │   ← hollow sites (center of 4 Cu)
    ·──B──·──B──·   ← bridge sites (midpoint between 2 Cu)
    │  H  │  H  │
    ·─────·─────·
    T = top site (directly above a Cu atom)
```

- **Top site**: CO prefers this on Cu(100)
- **Hollow site**: H prefers this on Cu(100)

## 3. VASP Input Files

### 3.1 INCAR — Clean Slab

```
# INCAR — Cu(100) clean slab geometry optimization
SYSTEM  = Cu100 clean slab
ISTART  = 0
ICHARG  = 2
ENCUT   = 400        # eV; converge with respect to this
PREC    = Accurate
EDIFF   = 1E-5
EDIFFG  = -0.02      # ionic convergence: max force < 0.02 eV/Ang
NSW     = 100
IBRION  = 2          # conjugate gradient
ISIF    = 2          # relax atoms only, fix cell shape
POTIM   = 0.3
ISMEAR  = 1          # Methfessel-Paxton for metals
SIGMA   = 0.2        # smearing width (eV)
ALGO    = Fast
NELM    = 100
LWAVE   = .TRUE.
LCHARG  = .TRUE.
```

### 3.2 KPOINTS

```
Automatic
0
Gamma
  4  4  1
  0  0  0
```
A 4×4×1 Gamma-centered mesh is standard for a (2×2) surface cell. Increase to 6×6×1 for production-quality results.

### 3.3 INCAR — Slab + CO (top site)

Append a CO molecule at the top site of the Cu(100) surface. Place C at ~1.9 Å above a surface Cu atom, O at ~3.1 Å (C–O bond ≈ 1.16 Å, CO stands upright).

```
# INCAR — Cu(100) + CO (top site, geometry optimization)
SYSTEM  = Cu100 + CO top site
ISTART  = 0
ICHARG  = 2
ENCUT   = 400
PREC    = Accurate
EDIFF   = 1E-5
EDIFFG  = -0.02
NSW     = 100
IBRION  = 2
ISIF    = 2
POTIM   = 0.3
ISMEAR  = 1
SIGMA   = 0.2
ALGO    = Fast
NELM    = 100
IVDW    = 12         # D3-BJ vdW correction — important for CO
LWAVE   = .TRUE.
LCHARG  = .TRUE.
```

### 3.4 INCAR — Slab + H (hollow site)

Place H at the hollow site (~1.0 Å above the surface plane).

```
# INCAR — Cu(100) + H (hollow site, geometry optimization)
SYSTEM  = Cu100 + H hollow site
ISTART  = 0
ICHARG  = 2
ENCUT   = 400
PREC    = Accurate
EDIFF   = 1E-5
EDIFFG  = -0.02
NSW     = 100
IBRION  = 2
ISIF    = 2
POTIM   = 0.3
ISMEAR  = 1
SIGMA   = 0.2
ALGO    = Fast
NELM    = 100
LWAVE   = .TRUE.
LCHARG  = .TRUE.
```

### 3.5 Gas-phase reference energies

Calculate isolated CO (spin-paired) and H₂ molecule in a large box:

```
# INCAR — CO gas phase (12x12x12 Å box, Gamma only)
SYSTEM  = CO gas phase
ISTART  = 0
ICHARG  = 2
ENCUT   = 400
PREC    = Accurate
EDIFF   = 1E-6
NSW     = 50
IBRION  = 2
ISIF    = 2
POTIM   = 0.3
ISMEAR  = 0          # Gaussian for molecule
SIGMA   = 0.01
ALGO    = Fast
LWAVE   = .FALSE.
LCHARG  = .FALSE.
```

For H, we use ½ E(H₂), where H₂ is calculated in the same large box with ISMEAR=0.

## 4. Adsorption Energy Calculation

After all VASP runs finish, extract the final total energies from OSZICAR or OUTCAR and compute:

$$E_{\rm ads}(\rm CO) = E_{\rm Cu100+CO} - E_{\rm Cu100} - E_{\rm CO(gas)}$$

$$E_{\rm ads}(\rm H) = E_{\rm Cu100+H} - E_{\rm Cu100} - \tfrac{1}{2}E_{\rm H_2(gas)}$$

The code below reads the energies from OUTCAR files (stored in `../tmp/Cu100/`) and computes the adsorption energies.

In [ ]:
from ase.io import read

# Read converged calculations
# Each directory should contain a completed VASP calculation (OUTCAR)
slab        = read('../tmp/Cu100/slab/OUTCAR',       index=-1)
slab_CO     = read('../tmp/Cu100/slab_CO/OUTCAR',    index=-1)
slab_H      = read('../tmp/Cu100/slab_H/OUTCAR',     index=-1)
CO_gas      = read('../tmp/Cu100/CO_gas/OUTCAR',     index=-1)
H2_gas      = read('../tmp/Cu100/H2_gas/OUTCAR',     index=-1)

E_slab    = slab.get_potential_energy()
E_slab_CO = slab_CO.get_potential_energy()
E_slab_H  = slab_H.get_potential_energy()
E_CO_gas  = CO_gas.get_potential_energy()
E_H2_gas  = H2_gas.get_potential_energy()

E_ads_CO = E_slab_CO - E_slab - E_CO_gas
E_ads_H  = E_slab_H  - E_slab - 0.5 * E_H2_gas

print(f'Clean slab energy :  {E_slab:.4f} eV')
print(f'Slab + CO energy  :  {E_slab_CO:.4f} eV')
print(f'Slab + H energy   :  {E_slab_H:.4f} eV')
print(f'CO gas energy     :  {E_CO_gas:.4f} eV')
print(f'H2 gas energy     :  {E_H2_gas:.4f} eV')
print()
print(f'E_ads(CO) = {E_ads_CO:.3f} eV  (top site)')
print(f'E_ads(H)  = {E_ads_H:.3f} eV  (hollow site)')

In [ ]:
import matplotlib.pyplot as plt

labels = ['CO (top)', 'H (hollow)']
energies = [E_ads_CO, E_ads_H]
colors = ['steelblue', 'coral']

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(labels, energies, color=colors, edgecolor='black', width=0.4)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_ylabel('Adsorption energy (eV)')
ax.set_title('CO vs. H adsorption on Cu(100)')
for bar, val in zip(bars, energies):
    ax.text(bar.get_x() + bar.get_width()/2, val - 0.05 * abs(val),
            f'{val:.2f} eV', ha='center', va='top', fontsize=11)
plt.tight_layout()
plt.show()

## 5. Discussion

### Why does CO bind at the top site?

CO binds at the **top (atop) site** on most close-packed metal surfaces including Cu(100). The C lone pair donates into the Cu 3d/4s states (Blyholder model: 5σ donation and 2π* back-donation). The overlap is maximized at the top site where a single Cu atom interacts directly with C.

> **PBE artifact**: PBE predicts CO preferring the hollow site on some metals (e.g., Pt(111)), which is the wrong answer. This 'CO adsorption site puzzle' was a famous DFT failure resolved by using the RPBE functional or hybrid functionals. On Cu(100), PBE gives the correct top-site preference.

### Why does H bind at the hollow site?

Hydrogen adsorbs as an atom (H₂ dissociates at the surface). The H 1s orbital interacts with multiple Cu atoms simultaneously, maximizing the bonding overlap in the **hollow** site. The adsorption energy is much larger in magnitude than CO because H forms a true chemisorptive bond.

### Practical implications

These adsorption energies are inputs to microkinetic models for catalytic reactions such as:
- CO₂ reduction → CO + O* → further reduction
- Water-gas shift reaction: CO + H₂O → CO₂ + H₂
- Fischer-Tropsch synthesis

## Homework

1. **A–I**: Calculate the adsorption energy of CO on the Cu(111) surface (top, bridge, and fcc-hollow sites). Which site is preferred? Compare with the Cu(100) result.
2. **J–R**: Calculate the adsorption energy of O on Cu(100) at the hollow site. Compare the O and H adsorption energies and explain the difference.
3. **S–Z**: Repeat the CO adsorption calculation on Cu(100) without the IVDW=12 tag and compare. How large is the vdW contribution to the CO adsorption energy?

## Further Reading

- Hammer, B.; Hansen, L. B.; Nørskov, J. K. *Phys. Rev. B* **59**, 7413 (1999) — RPBE and CO adsorption site puzzle
- Blyholder, G. *J. Phys. Chem.* **68**, 2772 (1964) — Blyholder model for CO–metal bonding
- Nørskov, J. K. et al. *J. Catal.* **209**, 275 (2002) — universality of adsorption energies
- [VASP Wiki: Surface calculations](https://www.vasp.at/wiki/Surface_calculations)
- [ASE surface tutorial](https://wiki.fysik.dtu.dk/ase/tutorials/surface.html)

---
*Ch121a | Caltech | Module 3 — Notebook 4b of 6*